## Initialization

In [ ]:
# Imports
# import pickle
# from pathlib import Path
from typing import Callable, Literal
from random import sample
from statistics import mean

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from scipy.signal import find_peaks
from scipy.interpolate import CubicSpline
from scipy.stats import linregress
from scipy.optimize import curve_fit
import pandas as pd
# from scipy.interpolate import interp1d
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
# from data_processing.processing.calibration import Detector, recalibrate
# from data_processing.processing.figure_of_merit import gaussian
# from data_processing.processing.neutron_classification import classify
# from data_processing.processing.neutron_window_generation import (
#     generate_nasa_neutron_window,
#     generate_n_distro_neutron_window
# )
from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
# from data_processing.arc_paths import (INPUT_DATA_FOLDER, get_exp_root,
#                                        get_parq_root)
from data_processing.dataframe_validation import DetectorDataframeColumn, EnergyColumn
from data_processing.experiment_data_keys import (ExperimentDataKey,
                                                  ExperimentNeutronData)
from data_processing.helpers import (
    get_input_with_default,
    # get_input_required,
    input_experiment_ids,
    stop
)
# from data_processing.loading import get_neutron_window_paths, load_side_borders
# from data_processing.loading.dataframe_loading import load_psd
# from data_processing.loading.timetag_processing import calculate_timetag_hours
# from data_processing.reporting import plot_classification
# from data_processing.types import (BimodalBounds, BimodalParams,
#                                    NasaGenerationSettings,
#                                    NeutronWindowSettings, WindowType)
# from scipy.optimize import curve_fit
# from scipy.signal import deconvolve
from data_processing.processing.figure_of_merit import gaussian

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

### Functions

In [ ]:
def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> proc_types.NasaGenerationSettings:
    sigma = get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee) (default)
2: newer (~0.1866 MeVee)
or press Enter for default
""",
            1,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = ExperimentDataKey.NASA_BORDERS if existing_left_border_version_input == 1 else ExperimentDataKey.NASA_BORDERS_RECALC
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = load.get_neutron_window_paths(file_name_prefix=file_name_prefix)
            left_border, _ = load.load_side_borders(side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.1966)
""",
            0.1966,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], proc.AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], proc.AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))


def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))

## Experiment ID Input

In [ ]:
# experiment_ids = input_experiment_ids()
experiment_ids = ["TB-leading_edge", "TB-26"]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

## Data Loading and Initial Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id, with_flags=True)
    exp_data["signals_df"] = load.load_parquet_signals(exp_id)

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = load.calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = proc.recalibrate(unclassified_df, proc.Detector.ZERO)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

## Pulse Selection

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    print(exp_id)
    # neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
    # gamma_only = exp_data[ExperimentDataKey.GAMMA_ONLY]
    signals_df = exp_data["signals_df"]
    # print(neutrons_only.shape)
    print(signals_df.shape)
    
    # neutron_signals = signals_df.loc[neutrons_only.index].astype("int32")
    neutron_signals = signals_df.astype("int32")
    # gamma_signals = signals_df.loc[gamma_only.index].astype("int32")
    print(neutron_signals.shape)

    n_signals_np = neutron_signals.to_numpy()
    n_baselines = n_signals_np.max(axis=1).reshape(-1, 1)
    n_signals_np = -n_signals_np + n_baselines
    print(n_signals_np.max())
    neutron_signals = pd.DataFrame(n_signals_np, index=neutron_signals.index, columns=neutron_signals.columns)
    # neutrons_only["peak_height"] = neutron_signals.max(axis=1)

    # g_signals_np = gamma_signals.to_numpy()
    # g_baselines = g_signals_np.max(axis=1).reshape(-1, 1)
    # g_signals_np = -g_signals_np + g_baselines
    # print(g_signals_np.max())
    # gamma_signals = pd.DataFrame(g_signals_np, index=gamma_signals.index, columns=gamma_signals.columns)

    exp_data["neutron_signals"] = neutron_signals
    # exp_data["gamma_signals"] = gamma_signals

## Plotting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"

In [ ]:
signals_count = 1000
fig, axs = plt.subplots(ncols=2, nrows=2, figsize=(14, 12))
axs_top, axs_bottom = axs
fig.subplots_adjust(wspace=0.3, hspace=0.25)
axs_bottom[0].sharey(axs_bottom[1])

for i, (exp_name, exp_data) in enumerate(experiment_neutron_data.items()):
    print(exp_name)
    neutron_signals = exp_data["neutron_signals"]
    all_max_x = []
    all_max_y = []
    for row_idx, row_data in neutron_signals.head(signals_count).iterrows():
        neutron_x = row_data.index.map(lambda x: int(x) * 2)
        neutron_y = row_data.values

        max_y = neutron_y.max()
        max_y_idx = neutron_y.argmax()
        max_x = neutron_x[max_y_idx]
        relevant_region = (100, 200) if exp_name == "TB-leading_edge" else (50, 150)
        rel_lo, rel_hi = relevant_region
        if rel_lo <= max_x < rel_hi:
            all_max_x.append(max_x)
            all_max_y.append(max_y)
        
        plotline = axs_top[i].plot(neutron_x, neutron_y, alpha=0.01)
        plot_color = plotline[0].get_color()
        # ax.vlines(max_x, 0, max_y, color=plot_color)
        if not (exp_name == "TB-leading_edge" and (max_x < 140 or max_x > 160)):
            axs_top[i].plot(max_x, max_y, ".", color="red", alpha=0.25)
            axs_top[i].plot(max_x, 0, ".", color="red")
        # if exp_name == "TB-leading_edge":
        #     axs_top[i].set_xlim(100, 200)
        # else:
        #     axs_top[i].set_xlim(50, 150)
        axs_top[i].set_xlim(rel_lo, rel_hi)
        axs_top[i].set_xlabel("Time (ns)", fontsize=fontsize)
        axs_top[i].set_ylabel("Pulse height (ADC channels x1000)", fontsize=fontsize)
        axs_top[i].tick_params(labelsize=fontsize)
        axs_top[i].yaxis.set_major_formatter(lambda x, pos: str(x // 1000))
        if exp_name == "TB-leading_edge":
            axs_top[i].set_title("Leading edge trigger", fontsize=fontsize)
        else:
            axs_top[i].set_title("CFD trigger", fontsize=fontsize)
    
    # axs_bottom[i].scatter(all_max_y, all_max_x, color="red")
    # if exp_name == "TB-leading_edge":
    #     axs_bottom[i].set_ylim(143, 155)
    # # else:
    # #     axs_bottom[i].set_xlim(90, 110)

    # best_fit_result = linregress(all_max_y, all_max_x)
    # print(best_fit_result)
    # print(best_fit_result.rvalue**2)
    # slope, intercept, *_ = best_fit_result
    # best_fit_x = np.linspace(0, max(all_max_y), 100)
    # best_fit_y = slope * best_fit_x + intercept
    # axs_bottom[i].plot(best_fit_x, best_fit_y, linewidth=3)
    # axs_bottom[i].set_xlabel("Pulse height (ADC channels x1000)", fontsize=fontsize)
    # axs_bottom[i].set_ylabel("Time (ns)", fontsize=fontsize)
    # axs_bottom[i].tick_params(labelsize=fontsize)
    # axs_bottom[i].xaxis.set_major_formatter(lambda x, pos: str(x // 1000))

    mean_max_x = mean(all_max_x)
    variance_max_x = [x - mean_max_x for x in all_max_x]
    # if exp_name == "TB-leading_edge":
    #     bins = np.arange(143, 156, 2)
    # else:
    #     bins = np.arange(93, 104, 2)
    # hist_counts, hist_bins = np.histogram(all_max_x, bins=bins)
    # if exp_name == "TB-leading_edge":
    #     bins = np.arange(-5, 46, 2)
    # else:
    #     bins = np.arange(93, 104, 2)
    bins = np.arange(-5, 7, 2)
    hist_counts, hist_bins = np.histogram(variance_max_x, bins=bins)
    bin_mids = (hist_bins[1:] + hist_bins[:-1]) / 2
    fit_params, fit_cov = curve_fit(gaussian, bin_mids, hist_counts)
    print(fit_params)
    print(np.sqrt(np.diag(fit_cov)))
    gauss_x = np.linspace(hist_bins[0], hist_bins[-1], 100)
    gauss_y = gaussian(gauss_x, *fit_params)
    axs_bottom[i].bar(bin_mids, hist_counts)
    axs_bottom[i].plot(gauss_x, gauss_y, color=bg_bluegrey, lw=3)
    axs_bottom[i].set_xlabel("Peak time variance (ns)", fontsize=fontsize)
    axs_bottom[i].set_ylabel("Counts", fontsize=fontsize)
    axs_bottom[i].tick_params(labelsize=fontsize)
    axs_bottom[i].grid(visible=True, axis="y")
    if exp_name == "TB-leading_edge":
        axs_bottom[i].set_xticks(bin_mids)

In [ ]:
input("Processing done, hit Enter to finish")
stop()